# MR-Hydra — LOPO Class-Weight and Threshold Validation
## TSS–HSS Pareto frontier on the SWAN-SF five-feature dataset

This notebook adapts the supplied MR-Hydra pipeline to the same validation strategy used in the supplied QUANT notebook.

It will:

1. use development partitions **1, 2, 3, and 5**;
2. fit the MR-Hydra feature transforms once and cache the transformed development matrix;
3. run **leave-one-partition-out (LOPO)** validation;
4. search several class-weight choices and probability thresholds;
5. summarize TSS, HSS, balanced accuracy, precision, recall, F1, FPR, and FAR;
6. construct the **mean LOPO TSS–HSS Pareto frontier**; and
7. save the validation tables, out-of-fold probabilities, and Pareto graph.

**Intentional stopping point:** this notebook does **not** load partition 4, refit a final classifier on all development partitions, or evaluate a final model. Choose the configuration after inspecting the validation tables and Pareto frontier.

> Runtime note: the MR-Hydra transform is the expensive step. The transformed development matrix is cached so the class-weight and threshold search does not repeatedly transform the raw time series.

## 1. Install dependencies

In [ ]:
# aeon 1.5.0 matches the supplied MR-Hydra notebook.
%pip install -q "aeon==1.5.0" joblib

## 2. Imports and configuration

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import platform
import time
from pathlib import Path

import aeon
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from IPython.display import display
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

from aeon.transformations.collection.convolution_based import (
    HydraTransformer,
    MultiRocket,
)

print("aeon version:", aeon.__version__)
print("scikit-learn version:", sklearn.__version__)
print("PyTorch version:", torch.__version__)
print("CPU cores:", os.cpu_count())

In [ ]:
# =============================
# Data and output paths
# =============================
DATA_DIR = Path(
    "/content/drive/MyDrive/solar_flare_forecasting/Data/5_features_standardized"
)
OUTPUT_DIR = DATA_DIR / "mr_hydra_5_feature_lopo_outputs"
CACHE_DIR = OUTPUT_DIR / "cache"

# =============================
# Expected development data
# =============================
FEATURES_TO_USE = [
    "TOTUSJH",
    "TOTBSQ",
    "TOTPOT",
    "TOTUSJZ",
    "ABSNJZH",
]
DEVELOPMENT_PARTITIONS = [1, 2, 3, 5]
FINAL_TEST_PARTITION = 4  # deliberately untouched in this notebook

# =============================
# Class-weight and threshold grid
# =============================
CLASS_WEIGHT_OPTIONS = [
    ("none", None),
    ("balanced", "balanced"),
    ("positive_5x", {0: 1, 1: 5}),
    ("positive_10x", {0: 1, 1: 10}),
]
THRESHOLDS = np.linspace(0.05, 0.95, 19)

# =============================
# MR-Hydra transform settings
# Retained from the supplied MR-Hydra notebook.
# =============================
HYDRA_N_KERNELS = 8
HYDRA_N_GROUPS = 64
MULTIROCKET_N_KERNELS = 840
MULTIROCKET_MAX_DILATIONS = 32
MULTIROCKET_FEATURES_PER_KERNEL = 4
REFERENCE_SAMPLES = 20_000
TRANSFORM_BATCH_SIZE = 1_024

# =============================
# Linear classifier settings
# =============================
TRAIN_EPOCHS = 2
CLASSIFIER_BATCH_SIZE = 4_096
PREDICT_BATCH_SIZE = 8_192
SGD_ALPHA = 1e-4
RANDOM_SEED = 42

# =============================
# Validation and caching
# =============================
DROP_LOW_VARIANCE_CASES = True
VARIANCE_THRESHOLD = 1e-7
VALIDATION_BATCH_SIZE = 50_000
CACHE_TRANSFORMED_FEATURES = True
REUSE_VALID_CACHE = True
SAVE_TRANSFORM_BUNDLE = True

np.random.seed(RANDOM_SEED)

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Development partitions:", DEVELOPMENT_PARTITIONS)
print("Untouched final test partition:", FINAL_TEST_PARTITION)
print("Class weights:", [name for name, _ in CLASS_WEIGHT_OPTIONS])
print("Thresholds:", THRESHOLDS)

## 3. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATA_DIR}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset directory found.")
print("Output directory ready:", OUTPUT_DIR)
print("Cache directory ready:", CACHE_DIR)

## 4. Load and validate the development tensor

Only the development split and its partition IDs are loaded. Partition 4 remains untouched.

In [ ]:
def require_file(data_dir: Path, filename: str) -> Path:
    path = data_dir / filename
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")
    return path


x_train_path = require_file(DATA_DIR, "X_train.npy")
y_train_path = require_file(DATA_DIR, "y_train.npy")
train_partition_path = require_file(DATA_DIR, "train_partition_id.npy")
feature_names_path = require_file(DATA_DIR, "selected_feature_names.json")

for name, path in {
    "X_train": x_train_path,
    "y_train": y_train_path,
    "train_partition_id": train_partition_path,
    "selected_feature_names": feature_names_path,
}.items():
    print(f"{name:24s} {path}")

In [ ]:
# Memory mapping prevents an immediate full copy of the input tensor.
X_train_raw = np.load(x_train_path, mmap_mode="r")
y_train_raw = np.load(y_train_path)
train_partition_raw = np.load(train_partition_path)

with open(feature_names_path, "r") as f:
    selected_feature_names = json.load(f)

print("Raw shapes and dtypes")
print("X_train:            ", X_train_raw.shape, X_train_raw.dtype)
print("y_train:            ", y_train_raw.shape, y_train_raw.dtype)
print("train_partition_id: ", train_partition_raw.shape, train_partition_raw.dtype)
print("Feature names:      ", selected_feature_names)

In [ ]:
def ensure_aeon_layout(X, feature_count):
    """Return a view in aeon layout: cases x channels x timepoints."""
    if X.ndim != 3:
        raise ValueError(f"Expected a 3D tensor, received shape {X.shape}")

    if X.shape[1] == feature_count:
        return X, "cases x channels x timepoints"
    if X.shape[2] == feature_count:
        return X.transpose(0, 2, 1), "cases x timepoints x channels -> transposed"

    raise ValueError(
        f"Could not locate the feature axis in tensor shape {X.shape}; "
        f"expected {feature_count} feature channels."
    )


if selected_feature_names != FEATURES_TO_USE:
    raise ValueError(
        "selected_feature_names.json does not match the required channel order.\n"
        f"Expected: {FEATURES_TO_USE}\n"
        f"Found:    {selected_feature_names}"
    )

X_train_aeon, train_layout = ensure_aeon_layout(
    X_train_raw, len(selected_feature_names)
)

if X_train_aeon.shape[0] != len(y_train_raw):
    raise ValueError("X_train and y_train are not aligned.")
if X_train_aeon.shape[0] != len(train_partition_raw):
    raise ValueError("X_train and train_partition_id are not aligned.")

observed_partitions = sorted(int(x) for x in np.unique(train_partition_raw))
if observed_partitions != DEVELOPMENT_PARTITIONS:
    raise ValueError(
        "Unexpected development partitions. "
        f"Expected {DEVELOPMENT_PARTITIONS}, found {observed_partitions}."
    )

print("Training layout:", train_layout)
print("Aeon development shape:", X_train_aeon.shape)
print("Development partitions:", observed_partitions)

## 6. Fit and cache the MR-Hydra feature transform

The supplied scalable MR-Hydra notebook combines:

- `HydraTransformer` features with Hydra's sparse square-root scaling; and
- `MultiRocket` features with standard scaling.

To mirror the QUANT workflow and keep LOPO practical, these unsupervised transforms are fit once on a reproducible stratified reference subset of the development data, then held fixed for every fold. The transformed development matrix is written batch-by-batch to a memory-mapped `.npy` cache.

In [ ]:
def stratified_reference_indices(y: np.ndarray, n_samples: int, seed: int) -> np.ndarray:
    """Take a reproducible reference subset while preserving both classes."""
    if n_samples >= len(y):
        return np.arange(len(y), dtype=np.int64)

    rng = np.random.default_rng(seed)
    selected = []

    for cls in np.unique(y):
        cls_idx = np.flatnonzero(y == cls)
        cls_n = max(1, round(n_samples * len(cls_idx) / len(y)))
        cls_n = min(cls_n, len(cls_idx))
        selected.append(rng.choice(cls_idx, size=cls_n, replace=False))

    indices = np.concatenate(selected)

    if len(indices) > n_samples:
        indices = rng.choice(indices, size=n_samples, replace=False)
    elif len(indices) < n_samples:
        remaining = np.setdiff1d(np.arange(len(y)), indices, assume_unique=False)
        extra = rng.choice(remaining, size=n_samples - len(indices), replace=False)
        indices = np.concatenate([indices, extra])

    rng.shuffle(indices)
    return indices.astype(np.int64, copy=False)


def to_numpy_2d(values) -> np.ndarray:
    """Convert aeon/PyTorch transform output to a float32 2-D NumPy array."""
    if torch.is_tensor(values):
        values = values.detach().cpu().numpy()
    values = np.asarray(values, dtype=np.float32)
    if values.ndim != 2:
        raise ValueError(f"Expected 2-D transformed features, got {values.shape}")
    return values


def fit_hydra_sparse_scaler(X_hydra: np.ndarray) -> dict[str, np.ndarray]:
    """Fit the sparse scaling used by aeon's Hydra classifier."""
    Z = np.sqrt(np.clip(X_hydra, 0, None)).astype(np.float32, copy=False)
    zero_fraction = np.mean(Z == 0, axis=0, dtype=np.float64)
    epsilon = np.power(zero_fraction, 4) + 1e-8
    mean = Z.mean(axis=0, dtype=np.float64)
    std = Z.std(axis=0, dtype=np.float64, ddof=1)
    scale = std + epsilon
    scale[~np.isfinite(scale) | (scale == 0)] = 1.0
    return {
        "mean": mean.astype(np.float32),
        "scale": scale.astype(np.float32),
    }


def transform_hydra_scaled(
    X_hydra: np.ndarray,
    params: dict[str, np.ndarray],
) -> np.ndarray:
    Z = np.sqrt(np.clip(X_hydra, 0, None)).astype(np.float32, copy=False)
    nonzero = Z != 0
    Z = ((Z - params["mean"]) * nonzero) / params["scale"]
    return Z.astype(np.float32, copy=False)


def index_hash(indices) -> str:
    return hashlib.sha256(
        np.asarray(indices, dtype=np.int64).tobytes()
    ).hexdigest()


def array_hash(values) -> str:
    return hashlib.sha256(np.asarray(values).tobytes()).hexdigest()


def file_signature(path: Path) -> dict:
    stat = path.stat()
    return {
        "name": path.name,
        "size_bytes": int(stat.st_size),
        "modified_time_ns": int(stat.st_mtime_ns),
    }


reference_idx = stratified_reference_indices(
    y_development,
    n_samples=min(REFERENCE_SAMPLES, len(y_development)),
    seed=RANDOM_SEED,
)

print("Reference subset size:", len(reference_idx))
print(
    "Reference class counts:",
    dict(zip(*np.unique(y_development[reference_idx], return_counts=True))),
)

In [ ]:
transform_bundle_path = CACHE_DIR / "mr_hydra_transform_bundle.joblib"
feature_cache_path = CACHE_DIR / "X_development_mr_hydra.npy"
feature_cache_metadata_path = CACHE_DIR / "X_development_mr_hydra_metadata.json"

expected_cache_metadata = {
    "features": FEATURES_TO_USE,
    "n_cases": int(len(y_development)),
    "case_index_hash": index_hash(train_original_indices),
    "label_hash": array_hash(y_development),
    "partition_id_hash": array_hash(partition_id),
    "reference_index_hash": index_hash(reference_idx),
    "hydra_n_kernels": HYDRA_N_KERNELS,
    "hydra_n_groups": HYDRA_N_GROUPS,
    "multirocket_n_kernels": MULTIROCKET_N_KERNELS,
    "multirocket_max_dilations": MULTIROCKET_MAX_DILATIONS,
    "multirocket_features_per_kernel": MULTIROCKET_FEATURES_PER_KERNEL,
    "random_seed": RANDOM_SEED,
    "source_x_train": file_signature(x_train_path),
    "source_y_train": file_signature(y_train_path),
    "source_partition_ids": file_signature(train_partition_path),
}


def metadata_matches(saved: dict, expected: dict) -> bool:
    return all(saved.get(key) == value for key, value in expected.items())


cache_is_valid = False
if (
    CACHE_TRANSFORMED_FEATURES
    and REUSE_VALID_CACHE
    and transform_bundle_path.exists()
    and feature_cache_path.exists()
    and feature_cache_metadata_path.exists()
):
    with open(feature_cache_metadata_path, "r") as f:
        saved_cache_metadata = json.load(f)
    cache_is_valid = metadata_matches(saved_cache_metadata, expected_cache_metadata)

if cache_is_valid:
    print("Loading the existing valid MR-Hydra transform and feature cache.")
    transform_bundle = joblib.load(transform_bundle_path)
    hydra = transform_bundle["hydra_transformer"]
    multirocket = transform_bundle["multirocket_transformer"]
    hydra_scaler_params = transform_bundle["hydra_scaler_params"]
    multirocket_scaler = transform_bundle["multirocket_scaler"]
    hydra_feature_count = int(transform_bundle["hydra_feature_count"])
    multirocket_feature_count = int(transform_bundle["multirocket_feature_count"])
    combined_feature_count = int(transform_bundle["combined_feature_count"])
    transform_fit_seconds = float(transform_bundle.get("transform_fit_seconds", np.nan))
    scaler_fit_seconds = float(transform_bundle.get("scaler_fit_seconds", np.nan))

    X_development_mr_hydra = np.load(feature_cache_path, mmap_mode="r")
    if X_development_mr_hydra.shape != (
        len(y_development),
        combined_feature_count,
    ):
        raise ValueError(
            "The cached transformed matrix has an unexpected shape: "
            f"{X_development_mr_hydra.shape}"
        )
else:
    print("Fitting the MR-Hydra transforms on the reference subset...")
    X_reference = np.asarray(X_development[reference_idx], dtype=np.float32)

    hydra = HydraTransformer(
        n_kernels=HYDRA_N_KERNELS,
        n_groups=HYDRA_N_GROUPS,
        n_jobs=-1,
        random_state=RANDOM_SEED,
    )
    multirocket = MultiRocket(
        n_kernels=MULTIROCKET_N_KERNELS,
        max_dilations_per_kernel=MULTIROCKET_MAX_DILATIONS,
        n_features_per_kernel=MULTIROCKET_FEATURES_PER_KERNEL,
        n_jobs=-1,
        random_state=RANDOM_SEED,
    )

    start = time.perf_counter()
    hydra.fit(X_reference)
    multirocket.fit(X_reference)
    transform_fit_seconds = time.perf_counter() - start

    print(f"Transform fitting completed in {transform_fit_seconds / 60:.2f} minutes.")

    start = time.perf_counter()
    Xh_reference = to_numpy_2d(hydra.transform(X_reference))
    Xm_reference = to_numpy_2d(multirocket.transform(X_reference))

    hydra_scaler_params = fit_hydra_sparse_scaler(Xh_reference)
    multirocket_scaler = StandardScaler(copy=False)
    multirocket_scaler.fit(Xm_reference)

    hydra_feature_count = int(Xh_reference.shape[1])
    multirocket_feature_count = int(Xm_reference.shape[1])
    combined_feature_count = hydra_feature_count + multirocket_feature_count
    scaler_fit_seconds = time.perf_counter() - start

    print("Hydra transformed features:", hydra_feature_count)
    print("MultiRocket transformed features:", multirocket_feature_count)
    print("Combined transformed features:", combined_feature_count)
    print(f"Scaler fitting completed in {scaler_fit_seconds / 60:.2f} minutes.")

    transform_bundle = {
        "hydra_transformer": hydra,
        "multirocket_transformer": multirocket,
        "hydra_scaler_params": hydra_scaler_params,
        "multirocket_scaler": multirocket_scaler,
        "feature_names": FEATURES_TO_USE,
        "hydra_feature_count": hydra_feature_count,
        "multirocket_feature_count": multirocket_feature_count,
        "combined_feature_count": combined_feature_count,
        "reference_indices": reference_idx,
        "transform_fit_seconds": transform_fit_seconds,
        "scaler_fit_seconds": scaler_fit_seconds,
        "settings": expected_cache_metadata,
    }

    if SAVE_TRANSFORM_BUNDLE or CACHE_TRANSFORMED_FEATURES:
        joblib.dump(transform_bundle, transform_bundle_path, compress=3)
        print("Saved transform bundle:", transform_bundle_path)

    del Xh_reference, Xm_reference, X_reference
    gc.collect()

In [ ]:
def transform_mr_hydra_batch(X_batch: np.ndarray) -> np.ndarray:
    """Generate scaled and concatenated MR-Hydra features for one input batch."""
    X_batch = np.ascontiguousarray(X_batch, dtype=np.float32)

    Xh = to_numpy_2d(hydra.transform(X_batch))
    Xh = transform_hydra_scaled(Xh, hydra_scaler_params)

    Xm = to_numpy_2d(multirocket.transform(X_batch))
    Xm = multirocket_scaler.transform(Xm).astype(np.float32, copy=False)

    return np.concatenate((Xh, Xm), axis=1).astype(np.float32, copy=False)


# Smoke test before the long cache-building step.
smoke_n = min(16, len(X_development))
Xt_smoke = transform_mr_hydra_batch(X_development[:smoke_n])
assert Xt_smoke.shape == (smoke_n, combined_feature_count)
assert np.isfinite(Xt_smoke).all()
print("Transformed batch smoke test passed:", Xt_smoke.shape)
del Xt_smoke
gc.collect()

if not cache_is_valid:
    print("Building the transformed development cache...")
    cache_start = time.perf_counter()

    X_cache_writer = np.lib.format.open_memmap(
        feature_cache_path,
        mode="w+",
        dtype=np.float32,
        shape=(len(y_development), combined_feature_count),
    )

    for start_idx in tqdm(
        range(0, len(y_development), TRANSFORM_BATCH_SIZE),
        desc="Transforming development data",
    ):
        stop_idx = min(start_idx + TRANSFORM_BATCH_SIZE, len(y_development))
        X_cache_writer[start_idx:stop_idx] = transform_mr_hydra_batch(
            X_development[start_idx:stop_idx]
        )

    X_cache_writer.flush()
    del X_cache_writer
    gc.collect()

    cache_build_seconds = time.perf_counter() - cache_start

    saved_cache_metadata = {
        **expected_cache_metadata,
        "hydra_feature_count": hydra_feature_count,
        "multirocket_feature_count": multirocket_feature_count,
        "combined_feature_count": combined_feature_count,
        "dtype": "float32",
        "cache_build_seconds": cache_build_seconds,
    }
    with open(feature_cache_metadata_path, "w") as f:
        json.dump(saved_cache_metadata, f, indent=2)

    print(f"Feature cache completed in {cache_build_seconds / 60:.2f} minutes.")

X_development_mr_hydra = np.load(feature_cache_path, mmap_mode="r")

if not np.isfinite(X_development_mr_hydra[: min(10_000, len(y_development))]).all():
    raise ValueError("The transformed feature cache contains non-finite values.")

print("Transformed development matrix:", X_development_mr_hydra.shape)
print("Transformed dtype:", X_development_mr_hydra.dtype)
print("Feature cache:", feature_cache_path)

## 7. Metric and classifier helpers

In [ ]:
def positive_class_scores(
    classifier,
    X,
    indices,
    positive_label=1,
    batch_size=PREDICT_BATCH_SIZE,
    description="Predicting probabilities",
):
    classifier_classes = list(classifier.classes_)
    if positive_label not in classifier_classes:
        raise ValueError(
            f"Positive label {positive_label} not found in {classifier_classes}"
        )
    positive_column = classifier_classes.index(positive_label)

    scores = np.empty(len(indices), dtype=np.float32)
    for output_start in tqdm(
        range(0, len(indices), batch_size),
        desc=description,
        leave=False,
    ):
        output_stop = min(output_start + batch_size, len(indices))
        batch_indices = indices[output_start:output_stop]
        X_batch = np.asarray(X[batch_indices], dtype=np.float32)
        scores[output_start:output_stop] = classifier.predict_proba(X_batch)[
            :, positive_column
        ]
    return scores


def labels_from_threshold(scores, threshold, dtype=None):
    predictions = np.where(scores >= threshold, positive_label, negative_label)
    return predictions.astype(dtype) if dtype is not None else predictions


def binary_metrics(y_true, y_pred, scores=None):
    true_positive_mask = np.asarray(y_true) == positive_label
    pred_positive_mask = np.asarray(y_pred) == positive_label

    tn, fp, fn, tp = confusion_matrix(
        true_positive_mask,
        pred_positive_mask,
        labels=[False, True],
    ).ravel()

    pod = tp / (tp + fn) if (tp + fn) else np.nan
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    far = fp / (tp + fp) if (tp + fp) else np.nan
    tss = pod - fpr if np.isfinite(pod) and np.isfinite(fpr) else np.nan

    hss_denominator = ((tp + fn) * (fn + tn)) + ((tp + fp) * (fp + tn))
    hss = (
        2 * ((tp * tn) - (fp * fn)) / hss_denominator
        if hss_denominator
        else np.nan
    )

    output = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_positive": precision_score(
            y_true, y_pred, pos_label=positive_label, zero_division=0
        ),
        "recall_positive": recall_score(
            y_true, y_pred, pos_label=positive_label, zero_division=0
        ),
        "f1_positive": f1_score(
            y_true, y_pred, pos_label=positive_label, zero_division=0
        ),
        "POD_recall": pod,
        "FPR": fpr,
        "FAR": far,
        "TSS": tss,
        "HSS": hss,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }

    if scores is not None:
        output["roc_auc"] = roc_auc_score(true_positive_mask, scores)
        output["average_precision_pr_auc"] = average_precision_score(
            true_positive_mask, scores
        )

    return output

In [ ]:
def resolve_class_weight_map(y_fold_train, class_weight):
    """Return explicit per-class weights for streaming partial_fit."""
    if class_weight is None:
        return {negative_label: 1.0, positive_label: 1.0}

    if class_weight == "balanced":
        values, counts = np.unique(y_fold_train, return_counts=True)
        n_samples = len(y_fold_train)
        n_classes = len(values)
        return {
            int(value): float(n_samples / (n_classes * count))
            for value, count in zip(values, counts)
        }

    return {int(key): float(value) for key, value in class_weight.items()}


def build_sgd_classifier(seed):
    return SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=SGD_ALPHA,
        fit_intercept=True,
        learning_rate="optimal",
        shuffle=False,  # batches are shuffled explicitly below
        random_state=seed,
        average=True,
    )


def train_classifier_streaming(
    X,
    y,
    train_indices,
    class_weight,
    seed,
):
    """Train one weighted SGD classifier from the transformed feature cache."""
    classifier = build_sgd_classifier(seed)
    weight_map = resolve_class_weight_map(y[train_indices], class_weight)
    rng = np.random.default_rng(seed)
    first_partial_fit = True
    epoch_history = []

    for epoch in range(TRAIN_EPOCHS):
        epoch_start = time.perf_counter()
        shuffled_indices = rng.permutation(train_indices)

        for start_idx in tqdm(
            range(0, len(shuffled_indices), CLASSIFIER_BATCH_SIZE),
            desc=f"Training epoch {epoch + 1}/{TRAIN_EPOCHS}",
            leave=False,
        ):
            stop_idx = min(
                start_idx + CLASSIFIER_BATCH_SIZE,
                len(shuffled_indices),
            )
            batch_indices = shuffled_indices[start_idx:stop_idx]
            X_batch = np.asarray(X[batch_indices], dtype=np.float32)
            y_batch = y[batch_indices]

            if weight_map == {negative_label: 1.0, positive_label: 1.0}:
                sample_weight = None
            else:
                sample_weight = np.where(
                    y_batch == positive_label,
                    weight_map[positive_label],
                    weight_map[negative_label],
                ).astype(np.float64, copy=False)

            if first_partial_fit:
                classifier.partial_fit(
                    X_batch,
                    y_batch,
                    classes=classes,
                    sample_weight=sample_weight,
                )
                first_partial_fit = False
            else:
                classifier.partial_fit(
                    X_batch,
                    y_batch,
                    sample_weight=sample_weight,
                )

        epoch_seconds = time.perf_counter() - epoch_start
        epoch_history.append({
            "epoch": epoch + 1,
            "seconds": epoch_seconds,
            "minutes": epoch_seconds / 60,
        })

    return classifier, weight_map, epoch_history

## 8. Run leave-one-partition-out validation

For each class-weight option, one development partition is held out at a time. The classifier is trained only on the other three partitions. Probability thresholds are evaluated on the held-out partition, and every development case receives exactly one out-of-fold probability for each class-weight option.

In [ ]:
fold_metric_rows = []
fold_fit_rows = []
oof_scores_by_weight = {}
training_history_rows = []

for weight_index, (weight_name, class_weight) in enumerate(CLASS_WEIGHT_OPTIONS):
    print("\n" + "=" * 80)
    print("Class weight:", weight_name, class_weight)

    oof_scores = np.full(len(y_development), np.nan, dtype=np.float32)

    for held_out_partition in DEVELOPMENT_PARTITIONS:
        fold_train_idx = np.flatnonzero(partition_id != held_out_partition)
        fold_validation_idx = np.flatnonzero(partition_id == held_out_partition)

        y_fold_train = y_development[fold_train_idx]
        y_fold_validation = y_development[fold_validation_idx]

        if len(np.unique(y_fold_train)) != 2:
            raise ValueError(
                f"Fold excluding partition {held_out_partition} does not contain both classes."
            )
        if len(np.unique(y_fold_validation)) != 2:
            raise ValueError(
                f"Held-out partition {held_out_partition} does not contain both classes."
            )

        print(
            f"\nHeld-out partition {held_out_partition}: "
            f"train={len(fold_train_idx):,}, "
            f"validation={len(fold_validation_idx):,}"
        )

        fold_seed = RANDOM_SEED + int(held_out_partition)
        fit_start = time.perf_counter()
        classifier, resolved_weight_map, epoch_history = train_classifier_streaming(
            X_development_mr_hydra,
            y_development,
            fold_train_idx,
            class_weight,
            seed=fold_seed,
        )
        fit_seconds = time.perf_counter() - fit_start

        for epoch_record in epoch_history:
            training_history_rows.append({
                "class_weight_name": weight_name,
                "held_out_partition": int(held_out_partition),
                **epoch_record,
            })

        validation_scores = positive_class_scores(
            classifier,
            X_development_mr_hydra,
            fold_validation_idx,
            positive_label=positive_label,
            batch_size=PREDICT_BATCH_SIZE,
            description=f"Partition {held_out_partition} probabilities",
        )
        oof_scores[fold_validation_idx] = validation_scores

        for threshold in THRESHOLDS:
            validation_pred = labels_from_threshold(
                validation_scores,
                threshold,
                dtype=y_fold_validation.dtype,
            )
            metrics = binary_metrics(
                y_fold_validation,
                validation_pred,
                scores=validation_scores,
            )
            fold_metric_rows.append({
                "class_weight_name": weight_name,
                "class_weight": str(class_weight),
                "resolved_class_weight_map": str(resolved_weight_map),
                "held_out_partition": int(held_out_partition),
                "threshold": float(threshold),
                "n_fold_train": int(len(fold_train_idx)),
                "n_fold_validation": int(len(fold_validation_idx)),
                "fold_train_positive": int(
                    (y_fold_train == positive_label).sum()
                ),
                "fold_validation_positive": int(
                    (y_fold_validation == positive_label).sum()
                ),
                "classifier_fit_seconds": float(fit_seconds),
                **metrics,
            })

        fold_fit_rows.append({
            "class_weight_name": weight_name,
            "class_weight": str(class_weight),
            "resolved_class_weight_map": str(resolved_weight_map),
            "held_out_partition": int(held_out_partition),
            "n_fold_train": int(len(fold_train_idx)),
            "n_fold_validation": int(len(fold_validation_idx)),
            "classifier_fit_seconds": float(fit_seconds),
        })

        print(f"Classifier fit time: {fit_seconds / 60:.2f} minutes")
        del classifier, validation_scores
        gc.collect()

    if not np.isfinite(oof_scores).all():
        missing_count = int((~np.isfinite(oof_scores)).sum())
        raise RuntimeError(
            f"LOPO validation failed to produce {missing_count:,} out-of-fold "
            f"scores for class weight {weight_name}."
        )

    oof_scores_by_weight[weight_name] = oof_scores

fold_metrics_df = pd.DataFrame(fold_metric_rows)
fold_fit_times_df = pd.DataFrame(fold_fit_rows)
training_history_df = pd.DataFrame(training_history_rows)

print("\nCompleted LOPO validation.")
print("Fold metric rows:", len(fold_metrics_df))
display(fold_fit_times_df)

## 9. Aggregate LOPO results and identify the Pareto frontier

The main selection view uses **equal-weight means across the four held-out partitions**:

- a configuration is Pareto-optimal when no other configuration has both a higher-or-equal mean TSS and a higher-or-equal mean HSS, with at least one strict improvement;
- the automatically highlighted candidate is the Pareto point with the smallest Euclidean distance to the ideal point `(TSS, HSS) = (1, 1)`;
- the full frontier and ranked table are retained so the final choice can be made manually.

In [ ]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision_positive",
    "recall_positive",
    "f1_positive",
    "POD_recall",
    "FPR",
    "FAR",
    "TSS",
    "HSS",
    "roc_auc",
    "average_precision_pr_auc",
]

group_columns = ["class_weight_name", "class_weight", "threshold"]
fold_summary_df = (
    fold_metrics_df
    .groupby(group_columns)[metric_columns]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)
fold_summary_df.columns = [
    column[0] if column[1] == "" else f"{column[0]}_{column[1]}"
    for column in fold_summary_df.columns
]

pooled_rows = []
for weight_name, class_weight in CLASS_WEIGHT_OPTIONS:
    scores = oof_scores_by_weight[weight_name]
    for threshold in THRESHOLDS:
        predictions = labels_from_threshold(
            scores,
            threshold,
            dtype=y_development.dtype,
        )
        pooled_metrics = binary_metrics(
            y_development,
            predictions,
            scores=scores,
        )
        pooled_rows.append({
            "class_weight_name": weight_name,
            "class_weight": str(class_weight),
            "threshold": float(threshold),
            **{f"pooled_{key}": value for key, value in pooled_metrics.items()},
        })

pooled_metrics_df = pd.DataFrame(pooled_rows)
cv_summary_df = fold_summary_df.merge(
    pooled_metrics_df,
    on=group_columns,
    how="inner",
)


def pareto_optimal_mask(frame, tss_column="TSS_mean", hss_column="HSS_mean"):
    """Return True for rows not dominated on both mean TSS and mean HSS."""
    values = frame[[tss_column, hss_column]].to_numpy(dtype=float)
    valid = np.isfinite(values).all(axis=1)
    mask = np.zeros(len(frame), dtype=bool)

    for row_index, (tss_value, hss_value) in enumerate(values):
        if not valid[row_index]:
            continue

        dominates_row = (
            valid
            & (values[:, 0] >= tss_value)
            & (values[:, 1] >= hss_value)
            & (
                (values[:, 0] > tss_value)
                | (values[:, 1] > hss_value)
            )
        )
        dominates_row[row_index] = False
        mask[row_index] = not dominates_row.any()

    return mask


cv_summary_df["TSS_HSS_gap"] = (
    cv_summary_df["TSS_mean"] - cv_summary_df["HSS_mean"]
).abs()
cv_summary_df["distance_to_ideal"] = np.sqrt(
    (1.0 - cv_summary_df["TSS_mean"]) ** 2
    + (1.0 - cv_summary_df["HSS_mean"]) ** 2
)
cv_summary_df["is_pareto_optimal"] = pareto_optimal_mask(cv_summary_df)

pareto_frontier_df = (
    cv_summary_df[cv_summary_df["is_pareto_optimal"]]
    .sort_values(
        [
            "distance_to_ideal",
            "TSS_HSS_gap",
            "TSS_mean",
            "HSS_mean",
            "pooled_TSS",
            "precision_positive_mean",
        ],
        ascending=[True, True, False, False, False, False],
    )
    .reset_index(drop=True)
)

if pareto_frontier_df.empty:
    raise RuntimeError("No finite Pareto-optimal TSS/HSS configurations were found.")

selected_row = pareto_frontier_df.iloc[0]
selected_weight_name = selected_row["class_weight_name"]
selected_threshold = float(selected_row["threshold"])
selected_class_weight = dict(CLASS_WEIGHT_OPTIONS)[selected_weight_name]

ranked_cv_df = (
    cv_summary_df
    .sort_values(
        [
            "is_pareto_optimal",
            "distance_to_ideal",
            "TSS_HSS_gap",
            "TSS_mean",
            "HSS_mean",
            "pooled_TSS",
            "precision_positive_mean",
        ],
        ascending=[False, True, True, False, False, False, False],
    )
    .reset_index(drop=True)
)

# Comparison only: the configuration selected by a TSS-first rule.
tss_only_row = (
    cv_summary_df
    .sort_values(
        ["TSS_mean", "HSS_mean", "pooled_TSS", "precision_positive_mean"],
        ascending=[False, False, False, False],
    )
    .iloc[0]
)

# Secondary diagnostic: smallest absolute TSS-HSS gap.
minimum_gap = cv_summary_df["TSS_HSS_gap"].min()
gap_sanity_candidates_df = cv_summary_df[
    np.isclose(cv_summary_df["TSS_HSS_gap"], minimum_gap)
].sort_values(
    ["TSS_mean", "HSS_mean", "pooled_TSS"],
    ascending=[False, False, False],
)
gap_sanity_row = gap_sanity_candidates_df.iloc[0]
gap_sanity_agrees = (
    gap_sanity_row["class_weight_name"] == selected_weight_name
    and np.isclose(float(gap_sanity_row["threshold"]), selected_threshold)
)

selection_comparison_df = pd.DataFrame([
    {
        "selection_method": "Pareto + closest to (1,1)",
        "class_weight_name": selected_row["class_weight_name"],
        "threshold": selected_row["threshold"],
        "TSS_mean": selected_row["TSS_mean"],
        "HSS_mean": selected_row["HSS_mean"],
        "TSS_HSS_gap": selected_row["TSS_HSS_gap"],
        "distance_to_ideal": selected_row["distance_to_ideal"],
        "balanced_accuracy_mean": selected_row["balanced_accuracy_mean"],
        "precision_positive_mean": selected_row["precision_positive_mean"],
        "f1_positive_mean": selected_row["f1_positive_mean"],
    },
    {
        "selection_method": "TSS-first comparison",
        "class_weight_name": tss_only_row["class_weight_name"],
        "threshold": tss_only_row["threshold"],
        "TSS_mean": tss_only_row["TSS_mean"],
        "HSS_mean": tss_only_row["HSS_mean"],
        "TSS_HSS_gap": tss_only_row["TSS_HSS_gap"],
        "distance_to_ideal": tss_only_row["distance_to_ideal"],
        "balanced_accuracy_mean": tss_only_row["balanced_accuracy_mean"],
        "precision_positive_mean": tss_only_row["precision_positive_mean"],
        "f1_positive_mean": tss_only_row["f1_positive_mean"],
    },
    {
        "selection_method": "Minimum-gap diagnostic",
        "class_weight_name": gap_sanity_row["class_weight_name"],
        "threshold": gap_sanity_row["threshold"],
        "TSS_mean": gap_sanity_row["TSS_mean"],
        "HSS_mean": gap_sanity_row["HSS_mean"],
        "TSS_HSS_gap": gap_sanity_row["TSS_HSS_gap"],
        "distance_to_ideal": gap_sanity_row["distance_to_ideal"],
        "balanced_accuracy_mean": gap_sanity_row["balanced_accuracy_mean"],
        "precision_positive_mean": gap_sanity_row["precision_positive_mean"],
        "f1_positive_mean": gap_sanity_row["f1_positive_mean"],
    },
])

print("Automatically highlighted Pareto candidate")
print("Class weight:", selected_weight_name, selected_class_weight)
print(f"Threshold: {selected_threshold:.2f}")
print(f"Mean fold TSS: {selected_row['TSS_mean']:.4f}")
print(f"Mean fold HSS: {selected_row['HSS_mean']:.4f}")
print(f"TSS-HSS gap: {selected_row['TSS_HSS_gap']:.4f}")
print(f"Distance to ideal (1,1): {selected_row['distance_to_ideal']:.4f}")
print(f"TSS standard deviation: {selected_row['TSS_std']:.4f}")
print(f"Pooled out-of-fold TSS: {selected_row['pooled_TSS']:.4f}")
print(f"Minimum-gap diagnostic agrees: {gap_sanity_agrees}")

print("\nSelection comparison")
display(selection_comparison_df)

columns_to_show = [
    "class_weight_name",
    "threshold",
    "is_pareto_optimal",
    "distance_to_ideal",
    "TSS_HSS_gap",
    "TSS_mean",
    "TSS_std",
    "TSS_min",
    "TSS_max",
    "HSS_mean",
    "HSS_std",
    "balanced_accuracy_mean",
    "precision_positive_mean",
    "recall_positive_mean",
    "f1_positive_mean",
    "FPR_mean",
    "FAR_mean",
    "pooled_TSS",
    "pooled_HSS",
]

print("\nPareto frontier, ranked by distance to (1,1)")
display(pareto_frontier_df[columns_to_show])

print("\nconfigurations under the balanced ranking")
display(ranked_cv_df[columns_to_show])

## 10. Plot the mean LOPO TSS–HSS Pareto frontier

In [ ]:
pareto_plot_df = pareto_frontier_df.sort_values(
    ["TSS_mean", "HSS_mean"]
).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(
    cv_summary_df["TSS_mean"],
    cv_summary_df["HSS_mean"],
    color="lightgray",
    edgecolor="none",
    alpha=0.75,
    s=45,
    label="All configurations",
    zorder=1,
)

plot_min = float(np.nanmin(
    cv_summary_df[["TSS_mean", "HSS_mean"]].to_numpy(dtype=float)
))
reference_start = max(-1.0, plot_min - 0.05)
ax.plot(
    [reference_start, 1.0],
    [reference_start, 1.0],
    linestyle="--",
    color="black",
    linewidth=1.2,
    label="TSS = HSS",
    zorder=2,
)

ax.plot(
    pareto_plot_df["TSS_mean"],
    pareto_plot_df["HSS_mean"],
    marker="o",
    color="tab:blue",
    linewidth=2,
    markersize=6,
    label="Pareto frontier",
    zorder=3,
)

ax.scatter(
    selected_row["TSS_mean"],
    selected_row["HSS_mean"],
    color="red",
    edgecolor="black",
    linewidth=0.8,
    s=150,
    label="Automatically highlighted candidate",
    zorder=5,
)

ax.scatter(
    1.0,
    1.0,
    marker="*",
    color="gold",
    edgecolor="black",
    linewidth=0.8,
    s=260,
    label="Ideal (1, 1)",
    zorder=5,
)

selected_label = (
    f"Highlighted: {selected_weight_name}, threshold={selected_threshold:.2f}\n"
    f"TSS={selected_row['TSS_mean']:.3f}, HSS={selected_row['HSS_mean']:.3f}"
)
ax.annotate(
    selected_label,
    xy=(selected_row["TSS_mean"], selected_row["HSS_mean"]),
    xytext=(12, -42),
    textcoords="offset points",
    arrowprops={"arrowstyle": "->", "linewidth": 1.0},
    fontsize=9,
    bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "alpha": 0.9},
)

ax.set_title("MR-Hydra Mean LOPO TSS–HSS Pareto Frontier")
ax.set_xlabel("Mean LOPO TSS")
ax.set_ylabel("Mean LOPO HSS")
ax.set_xlim(reference_start, 1.03)
ax.set_ylim(reference_start, 1.03)
ax.grid(alpha=0.25)
ax.legend(loc="best")
fig.tight_layout()

pareto_plot_path = OUTPUT_DIR / "mr_hydra_lopo_tss_hss_pareto_frontier.png"
fig.savefig(pareto_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved Pareto visualization to:", pareto_plot_path)

## 11. Compare the best threshold for each class weight and inspect the highlighted folds

In [ ]:
best_per_weight_rows = []

for weight_name, _ in CLASS_WEIGHT_OPTIONS:
    weight_candidates_df = cv_summary_df[
        cv_summary_df["class_weight_name"] == weight_name
    ].copy()
    weight_candidates_df["is_pareto_within_weight"] = pareto_optimal_mask(
        weight_candidates_df
    )

    selected_weight_row = (
        weight_candidates_df[weight_candidates_df["is_pareto_within_weight"]]
        .sort_values(
            [
                "distance_to_ideal",
                "TSS_HSS_gap",
                "TSS_mean",
                "HSS_mean",
                "pooled_TSS",
                "precision_positive_mean",
            ],
            ascending=[True, True, False, False, False, False],
        )
        .iloc[0]
    )
    best_per_weight_rows.append(selected_weight_row)

best_per_weight_df = (
    pd.DataFrame(best_per_weight_rows)
    .sort_values(
        ["distance_to_ideal", "TSS_HSS_gap", "TSS_mean"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

display(best_per_weight_df[columns_to_show])

selected_fold_metrics_df = (
    fold_metrics_df[
        (fold_metrics_df["class_weight_name"] == selected_weight_name)
        & np.isclose(fold_metrics_df["threshold"], selected_threshold)
    ]
    .sort_values("held_out_partition")
    .reset_index(drop=True)
)

selected_fold_columns = [
    "held_out_partition",
    "n_fold_train",
    "n_fold_validation",
    "fold_validation_positive",
    "threshold",
    "TSS",
    "HSS",
    "balanced_accuracy",
    "precision_positive",
    "recall_positive",
    "f1_positive",
    "FPR",
    "FAR",
    "TP",
    "TN",
    "FP",
    "FN",
    "classifier_fit_seconds",
]

print("Automatically highlighted configuration in each held-out partition")
display(selected_fold_metrics_df[selected_fold_columns])

## 12. Save validation results

Only validation artifacts and the reusable feature-transform cache are saved. No final classifier is fit or saved.

In [ ]:
def make_json_safe(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): make_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)


fold_metrics_path = OUTPUT_DIR / "mr_hydra_lopo_fold_threshold_metrics.csv"
fold_fit_times_path = OUTPUT_DIR / "mr_hydra_lopo_fold_fit_times.csv"
training_history_path = OUTPUT_DIR / "mr_hydra_lopo_training_history.csv"
cv_summary_path = OUTPUT_DIR / "mr_hydra_lopo_class_weight_threshold_summary.csv"
best_per_weight_path = OUTPUT_DIR / "mr_hydra_lopo_best_threshold_per_class_weight.csv"
selected_fold_metrics_path = OUTPUT_DIR / "mr_hydra_lopo_highlighted_configuration_by_fold.csv"
pareto_frontier_path = OUTPUT_DIR / "mr_hydra_lopo_tss_hss_pareto_frontier.csv"
selection_comparison_path = OUTPUT_DIR / "mr_hydra_lopo_selection_method_comparison.csv"
gap_candidates_path = OUTPUT_DIR / "mr_hydra_lopo_minimum_gap_candidates.csv"
selected_oof_predictions_path = OUTPUT_DIR / "mr_hydra_lopo_highlighted_oof_predictions.csv"
oof_probabilities_path = OUTPUT_DIR / "mr_hydra_lopo_oof_probabilities_by_weight.npz"
validation_summary_path = OUTPUT_DIR / "mr_hydra_lopo_validation_summary.json"

fold_metrics_df.to_csv(fold_metrics_path, index=False)
fold_fit_times_df.to_csv(fold_fit_times_path, index=False)
training_history_df.to_csv(training_history_path, index=False)
ranked_cv_df.to_csv(cv_summary_path, index=False)
best_per_weight_df.to_csv(best_per_weight_path, index=False)
selected_fold_metrics_df.to_csv(selected_fold_metrics_path, index=False)
pareto_frontier_df.to_csv(pareto_frontier_path, index=False)
selection_comparison_df.to_csv(selection_comparison_path, index=False)
gap_sanity_candidates_df.to_csv(gap_candidates_path, index=False)

selected_oof_scores = oof_scores_by_weight[selected_weight_name]
selected_oof_pred = labels_from_threshold(
    selected_oof_scores,
    selected_threshold,
    dtype=y_development.dtype,
)

selected_oof_predictions_df = pd.DataFrame({
    "original_training_index": train_original_indices,
    "partition_id": partition_id,
    "y_true": y_development,
    "positive_probability": selected_oof_scores,
    "y_pred": selected_oof_pred,
    "highlighted_threshold": selected_threshold,
    "highlighted_class_weight": selected_weight_name,
})
selected_oof_predictions_df.to_csv(selected_oof_predictions_path, index=False)

np.savez_compressed(
    oof_probabilities_path,
    **{name: scores for name, scores in oof_scores_by_weight.items()},
)

validation_summary = {
    "model_name": "Scalable MR-Hydra LOPO validation",
    "feature_transforms": ["aeon.HydraTransformer", "aeon.MultiRocket"],
    "linear_classifier": "sklearn.SGDClassifier(loss='log_loss')",
    "features": FEATURES_TO_USE,
    "data_dir": str(DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
    "development_partitions": DEVELOPMENT_PARTITIONS,
    "untouched_final_test_partition": FINAL_TEST_PARTITION,
    "validation_method": "leave-one-partition-out",
    "selection_criterion": (
        "Pareto frontier on mean fold TSS and mean fold HSS, followed by "
        "minimum Euclidean distance to the ideal point (1, 1)"
    ),
    "final_model_trained": False,
    "partition4_loaded_or_evaluated": False,
    "n_development_cases": int(len(y_development)),
    "development_partition_counts": {
        str(partition): int((partition_id == partition).sum())
        for partition in DEVELOPMENT_PARTITIONS
    },
    "class_weight_candidates": {
        name: str(value) for name, value in CLASS_WEIGHT_OPTIONS
    },
    "threshold_candidates": THRESHOLDS.tolist(),
    "automatically_highlighted_class_weight_name": selected_weight_name,
    "automatically_highlighted_class_weight": selected_class_weight,
    "automatically_highlighted_threshold": selected_threshold,
    "automatically_highlighted_cv_summary": selected_row.to_dict(),
    "automatically_highlighted_fold_metrics": selected_fold_metrics_df.to_dict(
        orient="records"
    ),
    "minimum_gap_diagnostic_agrees": bool(gap_sanity_agrees),
    "minimum_gap_diagnostic": gap_sanity_row.to_dict(),
    "tss_first_comparison": tss_only_row.to_dict(),
    "transform_settings": {
        "hydra_n_kernels": HYDRA_N_KERNELS,
        "hydra_n_groups": HYDRA_N_GROUPS,
        "multirocket_n_kernels": MULTIROCKET_N_KERNELS,
        "multirocket_max_dilations": MULTIROCKET_MAX_DILATIONS,
        "multirocket_features_per_kernel": MULTIROCKET_FEATURES_PER_KERNEL,
        "reference_samples": int(len(reference_idx)),
        "hydra_feature_count": int(hydra_feature_count),
        "multirocket_feature_count": int(multirocket_feature_count),
        "combined_feature_count": int(combined_feature_count),
    },
    "classifier_settings": {
        "train_epochs": TRAIN_EPOCHS,
        "classifier_batch_size": CLASSIFIER_BATCH_SIZE,
        "sgd_alpha": SGD_ALPHA,
        "random_seed": RANDOM_SEED,
    },
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "aeon": aeon.__version__,
        "scikit_learn": sklearn.__version__,
        "torch": torch.__version__,
    },
}

with open(validation_summary_path, "w") as f:
    json.dump(make_json_safe(validation_summary), f, indent=2)

print("Saved fold threshold metrics:      ", fold_metrics_path)
print("Saved fold fit times:              ", fold_fit_times_path)
print("Saved training history:            ", training_history_path)
print("Saved ranked validation summary:   ", cv_summary_path)
print("Saved best threshold per weight:   ", best_per_weight_path)
print("Saved highlighted fold metrics:    ", selected_fold_metrics_path)
print("Saved TSS-HSS Pareto frontier:     ", pareto_frontier_path)
print("Saved selection comparison:        ", selection_comparison_path)
print("Saved minimum-gap candidates:      ", gap_candidates_path)
print("Saved highlighted OOF predictions: ", selected_oof_predictions_path)
print("Saved all OOF probabilities:       ", oof_probabilities_path)
print("Saved validation summary:          ", validation_summary_path)
print("Saved Pareto graph:                ", pareto_plot_path)
print("\nNo final classifier was trained. Partition 4 remains untouched.")

---

## Stop here before final-model training

Use these outputs to choose the class weight and threshold:

- `mr_hydra_lopo_class_weight_threshold_summary.csv`
- `mr_hydra_lopo_best_threshold_per_class_weight.csv`
- `mr_hydra_lopo_tss_hss_pareto_frontier.csv`
- `mr_hydra_lopo_tss_hss_pareto_frontier.png`
- the per-fold table shown above

The red point is an automatic balanced recommendation, not a requirement. Inspect nearby Pareto points and the fold-to-fold variability before deciding which configuration to use for the final MR-Hydra model.